## Scenario 1: A single data scientist participating in an ML competition

This scenario demonstrates how an individual data scientist can use MLflow to track machine learning experiments on their local machine. This is a common setup for solo projects, hackathons, or competitions where collaboration and remote access are not required.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (stores experiment metadata in the `mlruns/` folder)
- **Artifacts store:** Local filesystem (stores model files and other artifacts in the same `mlruns/` folder)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved locally. You can explore and compare your experiments using the MLflow UI.

### How to use the MLflow UI
- You can launch the MLflow UI by running `mlflow ui` in your terminal, or by running the provided code cell in this notebook.
- The UI will be available at [http://localhost:5001](http://localhost:5001) (port 5000 is used by macOS AirPlay Receiver).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.

> **Tip:** This local setup is ideal for learning and prototyping. For team projects or production, you would typically use a remote tracking server and a more robust backend (e.g., a database and cloud storage).

In [25]:
%pip install mlflow
import mlflow

Note: you may need to restart the kernel to use updated packages.


In [26]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////Users/isabelwu/Desktop/MBDS25/Term%202%20/ML%20Ops/NYC-TAXIS/03-experiment-tracking/mlflow.db'


In [27]:
mlflow.search_experiments()

[<Experiment: artifact_location=('/Users/isabelwu/Desktop/MBDS25/Term 2 /ML '
  'Ops/NYC-TAXIS/03-experiment-tracking/mlruns/1'), creation_time=1772554754528, experiment_id='1', last_update_time=1772554754528, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('/Users/isabelwu/Desktop/MBDS25/Term 2 /ML '
  'Ops/NYC-TAXIS/03-experiment-tracking/mlruns/0'), creation_time=1772554678588, experiment_id='0', last_update_time=1772554678588, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

### Creating an experiment and logging a new run

In [28]:
%pip install seaborn

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

models = [
    ("LogisticRegression", LogisticRegression(C=1.0, random_state=42, max_iter=1000, solver="lbfgs")),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, random_state=42))
]

for model_name, model in models:
    with mlflow.start_run() as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        model.fit(X, y)
        y_pred = model.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log confusion matrix as labeled DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(model, name="models", input_example=input_example)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on Iris dataset")
        print(f"Logged run for {model_name}, accuracy={acc:.3f}")

Note: you may need to restart the kernel to use updated packages.


2026/03/03 20:06:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for LogisticRegression, accuracy=0.973


2026/03/03 20:07:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for RandomForestClassifier, accuracy=1.000


In [29]:
mlflow.search_experiments()

[<Experiment: artifact_location=('/Users/isabelwu/Desktop/MBDS25/Term 2 /ML '
  'Ops/NYC-TAXIS/03-experiment-tracking/mlruns/1'), creation_time=1772554754528, experiment_id='1', last_update_time=1772554754528, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('/Users/isabelwu/Desktop/MBDS25/Term 2 /ML '
  'Ops/NYC-TAXIS/03-experiment-tracking/mlruns/0'), creation_time=1772554678588, experiment_id='0', last_update_time=1772554678588, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [30]:
import subprocess
import sys

# Kill any existing MLflow UI process on port 5001
subprocess.run(["bash", "-c", "lsof -ti :5001 | xargs kill -9 2>/dev/null"], capture_output=True)

# Launch MLflow UI on port 5001 (port 5000 is taken by macOS AirPlay Receiver)
# Must pass --backend-store-uri matching the tracking URI used by this notebook
tracking_uri = mlflow.get_tracking_uri()
subprocess.Popen([
    sys.executable, '-m', 'mlflow', 'ui',
    '--port', '5001',
    '--backend-store-uri', tracking_uri
])
print(f"MLflow UI started with backend: {tracking_uri}")
print("Open http://localhost:5001 in your browser.")

MLflow UI started with backend: sqlite:////Users/isabelwu/Desktop/MBDS25/Term%202%20/ML%20Ops/NYC-TAXIS/03-experiment-tracking/mlflow.db
Open http://localhost:5001 in your browser.


### Interacting with the model registry

In [31]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [ ]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")

Registry store URI not provided. Using backend store URI.


[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)
INFO:     Started parent process [29969]
2026/03/03 20:07:17 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_trace_scorer
2026/03/03 20:07:17 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/03/03 20:07:17 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_session_scorer
2026/03/03 20:07:17 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/03/03 20:07:17 INFO mlflow.server.jobs.utils: Starting dedicated Huey consumer for periodic tasks
2026/03/03 20:07:18 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: f662b67c-a337-4d84

INFO:     127.0.0.1:54678 - "GET / HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:54678 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:54678 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
INFO:     127.0.0.1:54680 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:54678 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK


2026/03/03 20:07:23 INFO mlflow.server.jobs.utils: Registered online_scoring_scheduler periodic task (runs every 1 minute)
2026/03/03 20:07:23 INFO huey.consumer: Huey consumer started with 10 thread, PID 30216 at 2026-03-03 19:07:23.377269
2026/03/03 20:07:23 INFO huey.consumer: Huey consumer started with 5 thread, PID 30214 at 2026-03-03 19:07:23.377311
[2026-03-03 20:07:23,377] INFO:huey.consumer:MainThread:Huey consumer started with 10 thread, PID 30216 at 2026-03-03 19:07:23.377269
[2026-03-03 20:07:23,377] INFO:huey.consumer:MainThread:Huey consumer started with 5 thread, PID 30214 at 2026-03-03 19:07:23.377311
2026/03/03 20:07:23 INFO huey.consumer: Scheduler runs every 1 second(s).
[2026-03-03 20:07:23,377] INFO:huey.consumer:MainThread:Scheduler runs every 1 second(s).
2026/03/03 20:07:23 INFO huey.consumer: Scheduler runs every 1 second(s).
[2026-03-03 20:07:23,377] INFO:huey.consumer:MainThread:Scheduler runs every 1 second(s).
2026/03/03 20:07:23 INFO huey.consumer: Periodi

INFO:     127.0.0.1:54678 - "GET /static-files/static/js/8799.5c4e7066.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54682 - "GET /static-files/static/js/548.ba82e795.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54680 - "GET /static-files/static/js/2365.07e09cf7.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54697 - "GET /static-files/static/css/6589.fd7db8ca.chunk.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:54699 - "GET /static-files/static/js/6589.6e623b4c.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54680 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:54701 - "POST /graphql HTTP/1.1" 200 OK
INFO:     127.0.0.1:54701 - "GET /static-files/static/js/5683.15e3f7e5.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54699 - "GET /static-files/static/js/8614.a3d7c5a1.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54697 - "GET /static-files/static/js/5714.8d7ff8c3.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:54701 - "GET /static-files/static/js/291.bd621101.chunk.js 

2026/03/03 20:07:37 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('my-experiment-1', 'default', '', 'active', 1772564857496, 1772564857496)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/.venv/lib/python3.13/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/.venv/lib/python3.13/site-packages/sqlalchemy/engine/default.py", line 952, in do_execute
    curs

INFO:     127.0.0.1:54763 - "POST /api/2.0/mlflow/experiments/create HTTP/1.1" 503 Service Unavailable


2026/03/03 20:07:49 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('my-experiment-1', 'default', '', 'active', 1772564869809, 1772564869809)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/.venv/lib/python3.13/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/isabelwu/Desktop/MBDS25/Term 2 /ML Ops/.venv/lib/python3.13/site-packages/sqlalchemy/engine/default.py", line 952, in do_execute
    curs

INFO:     127.0.0.1:54774 - "POST /api/2.0/mlflow/experiments/create HTTP/1.1" 503 Service Unavailable


2026/03/03 20:08:18 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 09d12a98-9754-46bc-bf78-de942ab9aaea.
[2026-03-03 20:08:18,076] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 09d12a98-9754-46bc-bf78-de942ab9aaea.
2026/03/03 20:08:23 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 6403523a-2bcb-46ac-b516-9d4d8ab0c8c7.
[2026-03-03 20:08:23,381] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 6403523a-2bcb-46ac-b516-9d4d8ab0c8c7.
2026/03/03 20:08:26 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 09d12a98-9754-46bc-bf78-de942ab9aaea
[2026-03-03 20:08:26,627] INFO:huey:Worker-4:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 09d12a98-9754-46bc-bf78-de942ab9aaea
2026/03/03 20:08:26 INFO huey: mlflow.serv